# ODI to Databricks Migration: Simple HR_TRG_DEP Insert
_Converted on 2024-07-30 12:00:00_

This notebook processes data from `HR.DEPARTMENTS` into `HR.TRG_DEP`.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "F")
dbutils.widgets.text("DATASOURCE_NUM_ID", "1")
dbutils.widgets.text("ETL_PROC_WID", "12345")
dbutils.widgets.text("ODI_SESS_NO", "987654")
dbutils.widgets.text("ETL_LAST_EXTRACT_TIME", "1900-01-01 00:00:00")
dbutils.widgets.text("ETL_CURRENT_EXTRACT_TIME", "2999-12-31 23:59:59")

## ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS
SELECT to_timestamp('${ETL_LAST_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_last_extract_time;

CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS
SELECT to_timestamp('${ETL_CURRENT_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_current_extract_time;

In [ ]:
display(spark.sql("""
  SELECT
    '${ETL_JOB_TYPE}' AS ETL_JOB_TYPE,
    CAST(${DATASOURCE_NUM_ID} AS BIGINT) AS DATASOURCE_NUM_ID,
    CAST(${ETL_PROC_WID} AS BIGINT) AS ETL_PROC_WID,
    '${ODI_SESS_NO}' AS ODI_SESS_NO,
    etl_last_extract_time AS ETL_LAST_EXTRACT_TIME,
    etl_current_extract_time AS ETL_CURRENT_EXTRACT_TIME
  FROM v_etl_last_extract_time, v_etl_current_extract_time
"""))

## Target Table: `workspace.hr.trg_dep`

In [ ]:
%sql
-- SCEN_TASK_NO in {10}, {20}, {30}
CREATE TABLE IF NOT EXISTS workspace.hr.trg_dep (
    DEPARTMENT_ID   BIGINT,
    DEPARTMENT_NAME STRING,
    MANAGER_ID      BIGINT,
    LOCATION_ID     BIGINT
) USING DELTA;

In [ ]:
%sql
INSERT INTO workspace.hr.trg_dep
  (
    DEPARTMENT_ID ,
    DEPARTMENT_NAME ,
    MANAGER_ID ,
    LOCATION_ID
  )
SELECT
  departments.DEPARTMENT_ID ,
  departments.DEPARTMENT_NAME ,
  departments.MANAGER_ID ,
  departments.LOCATION_ID
FROM
  workspace.hr.departments AS departments;

In [ ]:
%sql
SELECT COUNT(*) FROM workspace.hr.trg_dep;

## Cleanup

## Validation

In [ ]:
%sql
SELECT * FROM workspace.hr.trg_dep LIMIT 10;

## Conversion Notes
- `HR` schema converted to `workspace.hr`.
- Table names `TRG_DEP` and `DEPARTMENTS` converted to lowercase `trg_dep` and `departments`.
- Oracle hint `/*+ APPEND PARALLEL */` removed.
- Data types for `trg_dep` columns (DEPARTMENT_ID, MANAGER_ID, LOCATION_ID) inferred as `BIGINT` and (DEPARTMENT_NAME) as `STRING`.
- This script performs a direct `INSERT` into the target table, assuming data is appended or that external logic handles idempotency or pre-clearing of the target table. If a full refresh is intended (i.e., clearing `trg_dep` before insertion), a `TRUNCATE TABLE` or `DELETE FROM` statement would need to be added explicitly.